# 阶段二：数据清洗与特征工程

在这里，我们将读取之前爬取到的 1909 条微博原始数据，提取出它们的模式，将其转变为可用于机器学习的特征。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体，防止画图标注时中文变成方块乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print("完美！工具库加载成功！")

完美！工具库加载成功！


## 1. 读取我们辛勤采集的第一阶度数据
执行下方的单元格，Pandas 会帮我们像 Excel 一样把 CSV 文件读进内存，并展示最开头的3条数据。

In [2]:
data_path = 'weibo-search/结果文件/中东局势彻底失控/中东局势彻底失控_特征完整版.csv'
df = pd.read_csv(data_path)
df.rename(columns={'ͷurl': '头像url', 'ûǳ': '用户昵称'}, inplace=True, errors='ignore')
import re

# 显示总共有多少行、多少列
print(f"数据读取成功！一共包含 {df.shape[0]} 行， {df.shape[1]} 列。")

# 预览最开头的三条数据
df.head(3)

数据读取成功！一共包含 770 行， 40 列。


,id,bid,user_id,用户昵称,微博正文,头条文章url,发布位置,艾特用户,话题,转发数,...,is_default_avatar,is_random_name,daily_post_rate,total_engagement,zero_engagement,is_verified,sentiment_score,source,is_bot_pred,bot_probability
0,5272519483327893,QuqyUdXJH,1627888924,独孤九鎗,劝和促谈守正义中方紧急发声力阻中东战火蔓延2026年3月3日，中共中央政治局委员、外交部长王...,NaN,NaN,NaN,"秒懂热点就用智搜,微博智搜内容共创计划,王毅和以色列外长通电话",2,...,1,0,0.0,32,0,1,1.000000,iPhone客户端,0,0.0
1,5272518290050214,QuqwZ0d3U,1652933705,幸运的简单快乐-快乐简单,#日本政府改口#日本沉默背后的“经济理性”与“安全悖论”当全球目光聚焦于美以袭击伊朗引发的道...,NaN,NaN,NaN,"日本政府改口,美以军事打击伊朗",0,...,1,0,0.0,46,0,1,1.000000,iPhone 17 Pro Max,0,0.0
2,5272515807286408,QuqsYuzwI,1738004582,纵览新闻,【对话在以色列的河北人：#在以做厨师男子称暂没回国打算#】#在以同胞讲述见导弹飞过听到爆炸#...,NaN,NaN,NaN,"在以做厨师男子称暂没回国打算,在以同胞讲述见导弹飞过听到爆炸,中东局势彻底失控,王毅和以色列...",1,...,1,0,0.0,20,0,0,0.999998,微博视频号,0,0.0


## 2. 数据清洗 (Data Cleaning)

刚才爬回来的数据有很多杂质。在这个单元格中，我们需要自动化地：
- **删掉重复的微博**：网络请求翻页重叠会导致抓到重复数据，以 `id` 为基准去重。
- **填补空缺值**：有的博主没有填个人简介，有的也没有点赞。缺失的文本填成空白，缺失的数字全当 `0` 来处理。

In [3]:
# 1. 剔除重复的抓取（避免微博防爬翻页经常会吐出重复的）
df.drop_duplicates(subset=['id'], inplace=True)
print(f"清洗掉爬虫翻页重叠后，剩余独立微博数：{len(df)} 条")

# 2. 补全缺失文本数据
# 把博主为空的简介全部填入空白字符
df['description'] = df.get('description', pd.Series(['']*len(df))).fillna('')

# 3. 将粉丝、关注等量纲数据强制转型为纯数字，如果有缺失的一律按 0 处理
num_cols = ['followers_count', 'friends_count', 'statuses_count', 'reposts_count', 'comments_count', 'attitudes_count']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("\n非常好！最脏乱差的初始数据清洗完毕，接下来我们要提取高级指标了！")
df.head(3)

清洗掉爬虫翻页重叠后，剩余独立微博数：770 条

非常好！最脏乱差的初始数据清洗完毕，接下来我们要提取高级指标了！


,id,bid,user_id,用户昵称,微博正文,头条文章url,发布位置,艾特用户,话题,转发数,...,is_random_name,daily_post_rate,total_engagement,zero_engagement,is_verified,sentiment_score,source,is_bot_pred,bot_probability,description
0,5272519483327893,QuqyUdXJH,1627888924,独孤九鎗,劝和促谈守正义中方紧急发声力阻中东战火蔓延2026年3月3日，中共中央政治局委员、外交部长王...,NaN,NaN,NaN,"秒懂热点就用智搜,微博智搜内容共创计划,王毅和以色列外长通电话",2,...,0,0.0,32,0,1,1.000000,iPhone客户端,0,0.0,
1,5272518290050214,QuqwZ0d3U,1652933705,幸运的简单快乐-快乐简单,#日本政府改口#日本沉默背后的“经济理性”与“安全悖论”当全球目光聚焦于美以袭击伊朗引发的道...,NaN,NaN,NaN,"日本政府改口,美以军事打击伊朗",0,...,0,0.0,46,0,1,1.000000,iPhone 17 Pro Max,0,0.0,
2,5272515807286408,QuqsYuzwI,1738004582,纵览新闻,【对话在以色列的河北人：#在以做厨师男子称暂没回国打算#】#在以同胞讲述见导弹飞过听到爆炸#...,NaN,NaN,NaN,"在以做厨师男子称暂没回国打算,在以同胞讲述见导弹飞过听到爆炸,中东局势彻底失控,王毅和以色列...",1,...,0,0.0,20,0,0,0.999998,微博视频号,0,0.0,


## 3. 特征工程 (Feature Engineering)

这是甄别水军机器人的**核心步骤**！现在的数字和文本还太原始了，机器学习看不懂。我们需要用 Pandas 把它们变成机器能看懂的“特征维度”。

### 重点寻找水军的嫌疑特征
**特征 1：畸形的粉丝关注比**
正常人可能会有几百个关注，几十个粉丝。而量产拿来刷热度的“水军机器人”往往会**“疯狂批量关注上千人，但自己由于不发生活内容，所以基本是0粉丝”**。
所以我们要在这 1672 条数据右边，人为算出一个新列：`粉丝/关注比 (Follower-Friend Ratio)`。

In [4]:
# 提取特征 1：粉丝关注比
# 如果关注数为0（防止除以0的数学报错），我们直接把比值设为粉丝数本身
df['follower_friend_ratio'] = df.apply(
    lambda row: row['followers_count'] / row['friends_count'] if row['friends_count'] > 0 else row['followers_count'],
    axis=1
)

# 看看我们提取出来的新特征长什么样，随机挑5个博主看看
df[['用户昵称', 'followers_count', 'friends_count', 'follower_friend_ratio']].sample(5)

,用户昵称,followers_count,friends_count,follower_friend_ratio
272,网络醉江南,88276,1970,44.810152
156,AUTO老徐,1016854,898,1132.354120
536,津云新闻,11314302,3445,3284.267634
299,闻览红星新媒体中心,101225,114,887.938596
716,ZL_0329,10599,1564,6.776854


## 3.2 提取基于内容的嫌疑特征

另外一个很常见的机器人特征是：要么**只发非常短的口水话**（比如“哈哈哈哈”、“支持”），要么**乱用感叹号和问号**来吸引眼球。
所以我们要针对文本算出两个新纬度：
- `text_len` (正文长度)
- `exclamation_density` (感叹号密度，即感叹号数量占全文长度的比例)
- `has_link` (是否包含外链)

In [5]:
# 提取特征 2 & 3：正文长度与感叹号密度
# 确保文本都是字符串格式
df['微博正文'] = df['微博正文'].astype(str)

# 1. 计算文本长度
df['text_len'] = df['微博正文'].apply(len)

# 2. 计算感叹号密度 (包含中英文感叹号)
def calc_exclamation_density(text):
    if len(text) == 0:
        return 0.0
    exc_count = text.count('!') + text.count('！')
    return exc_count / len(text)

df['exclamation_density'] = df['微博正文'].apply(calc_exclamation_density)

# 3. 提取特征 4：判断是否包含 URL 外链 (很多水军的任务就是发广告链接)
# 利用正则表达式检查是否含有 http
df['has_link'] = df['微博正文'].str.contains('http', case=False, na=False).astype(int)

# 提取包含感叹号最多、或者最短的5条微博看看特征效果
df[['用户昵称', '微博正文', 'text_len', 'exclamation_density', 'has_link']].sort_values(by='exclamation_density', ascending=False).head(5)

# 4. 新增高级水军特征 (我们在底层爬虫已经把它们抓取到了)
# 默认头像、随机数字网名
df['is_default_avatar'] = df['头像url'].astype(str).apply(lambda x: 1 if 'default' in x.lower() or 'tvax' not in x else 0)
df['is_random_name'] = df['用户昵称'].astype(str).apply(lambda x: 1 if re.search(r'\d{5,}', x) else 0)

# 日均活跃发帖率
from datetime import datetime
today = datetime.now()
def get_daily_rate(row):
    try:
        if not row.get('account_created_at'): return 0.0
        dt = pd.to_datetime(row['account_created_at'])
        days_alive = (today - dt.replace(tzinfo=None)).days
        if days_alive <= 0: days_alive = 1
        return row.get('statuses_count', 0) / days_alive
    except Exception:
        return 0.0
df['daily_post_rate'] = df.apply(get_daily_rate, axis=1)

# 零互动概率（发出来的东西根本没人看）
df['total_engagement'] = df.get('转发数', 0) + df.get('评论数', 0) + df.get('点赞数', 0)
df['zero_engagement'] = df['total_engagement'].apply(lambda x: 1 if x == 0 else 0)

# 认证大V特征
df['is_verified'] = df.get('user_authentication', df.get('会员类型', pd.Series(['']*len(df)))).astype(str).apply(lambda x: 1 if 'V' in x or '认证' in x else 0)


## 4. 寻找真相：数据打标 (Data Labeling)

这是本阶段的最后一步，也是最至关重要的一步！机器学习算法本质上是个学习算式的学生，它需要你先塞给它一份有“标准答案”的卷子它才能看懂。
我们需要在表的最后增加一列 `is_bot` （是不是机器人：1代表是，0代表正常人）。

在真实的互联网大厂中，这需要专门的“审核标注团队”盯着上万条数据手动打标签。但在这里，我们用一种聪明的毕设进阶做法：**“弱监督启发式规则”**。
我们利用刚才提炼出的特征，用一套严苛的条件自动筛选出一批高度嫌疑人：
1. **没粉丝还疯狂关注别人的潜水号**（如粉丝<10，关注>500）
2. **引流营销号**（没多少粉丝且正文包含 http 外链）
3. **带节奏的情绪号**（感叹号密度大于 8%）

In [6]:
# 定义打标“过滤器” (加入对高级特征的支持)
def label_bot(row):
    # 规则1：关注数远大于粉丝数 (例如关注>500且粉丝<10)
    if row.get('friends_count', 0) > 500 and row.get('followers_count', 0) < 10:
        return 1
    # 规则2：带外链且粉丝很少的引流号
    if row.get('has_link', 0) == 1 and row.get('followers_count', 0) < 100:
        return 1
    # 规则3：感叹号等带节奏标点密度过高的情绪机器
    if row.get('exclamation_density', 0.0) > 0.08:
        return 1
    # 新增规则4：使用默认无头像 + 粉丝少
    if row.get('is_default_avatar', 0) == 1 and row.get('followers_count', 0) < 50:
        return 1
    # 新增规则5：名字是随机长数字串
    if row.get('is_random_name', 0) == 1:
        return 1
    # 新增规则6：高频发水贴的机器狗 (日均发帖>20条且通常0互动)
    if row.get('daily_post_rate', 0.0) > 20 and row.get('zero_engagement', 0) == 1:
        return 1
    # 豁免规则：如果是实名认证大V或者金V，基本上不可能是廉价机器水军
    if row.get('is_verified', 0) == 1:
        return 0
    
    # 不满足以上极端情况的小白，我们先假定他是正常人类
    return 0

# 运用这套升级版的火眼金睛给每一行打上标签！
df['is_bot'] = df.apply(label_bot, axis=1)

bot_count = df['is_bot'].sum()
print(f"自动化打标完毕！\n>> 我们在这 {len(df)} 人的池子里，利用高阶特征抓出了 {bot_count} 个高度疑似机器狗的水军。")

final_data_path = 'weibo-search/结果文件/中东局势彻底失控/machine_learning_ready.csv'
df.to_csv(final_data_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 第二阶段大功告成！高精度的最终数据大餐已做好，文件位于：{final_data_path}")


自动化打标完毕！
>> 我们在这 770 人的池子里，利用高阶特征抓出了 10 个高度疑似机器狗的水军。

✅ 第二阶段大功告成！高精度的最终数据大餐已做好，文件位于：weibo-search/结果文件/中东局势彻底失控/machine_learning_ready.csv
